In [ ]:
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
import os, base64
load_dotenv(override=True)
print("Endpoint:", os.environ["AZURE_OPENAI_AUDIO_ENDPOINT"])
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)
client = OpenAI(
    base_url=os.environ["AZURE_OPENAI_AUDIO_ENDPOINT"],
    api_key=token_provider,
)

# Read and encode audio file  
with open('analysis.wav', 'rb') as wav_reader: 
    encoded_string = base64.b64encode(wav_reader.read()).decode('utf-8') 

# Make the audio chat completions request
completion = client.chat.completions.create( 
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"], 
    modalities=["text", "audio"], 
    audio={"voice": "alloy", "format": "wav"}, 
    messages=[ 
        { 
            "role": "user", 
            "content": [ 
                {  
                    "type": "text", 
                    "text": "Provide a word for word transcript of this audio clip." 
                }, 
                { 
                    "type": "input_audio", 
                    "input_audio": { 
                        "data": encoded_string, 
                        "format": "wav" 
                    } 
                } 
            ] 
        }, 
    ] 
) 
print("prompt  'Provide a word for word transcript of this audio clip.'")
print(completion.choices[0].message.audio.transcript)

# Write the output audio data to a file
wav_bytes = base64.b64decode(completion.choices[0].message.audio.data)
with open("analysis.wav", "wb") as f:
    f.write(wav_bytes)

Endpoint: https://jacwang-1123-resource.openai.azure.com/openai/v1


BadRequestError: Error code: 400 - {'error': {'message': 'The requested operation is unsupported.'}}